In [17]:
%run ../setup/01_config

Box(children=(Label(value='Catalog'), Text(value='dbr_dev')))

Box(children=(Label(value='Schema'), Text(value='bronze')))

In [18]:
from pyspark.sql import Window
from pyspark.sql.functions import col, current_timestamp, lit, row_number, desc



In [19]:
BRONZE_MENU_TABLE = f"{CATALOG}.{SCHEMA}.brz_menu_items"
SILVER_MENU_TABLE = f"{CATALOG}.{SCHEMA}.slv_menu_items"

In [20]:
df_bronze_menu = spark.read.table(BRONZE_MENU_TABLE)

w = Window.partitionBy("menu_item_id").orderBy(desc("_ingestion_timestamp"))

df_menu_typed = (
    df_bronze_menu
    .select(
        col("menu_item_id").cast("int"),
        col("item_name").cast("string"),
        col("price").cast("double"),
        col("_ingestion_timestamp"),
    )
    .withColumn("rn", row_number().over(w))
    .filter("rn = 1")
    .drop("rn", "_ingestion_timestamp")
)

df_menu_typed.createOrReplaceTempView("menu_updates")

In [21]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_MENU_TABLE} (
    menu_item_id INT,
    item_name STRING,
    price DOUBLE,
    valid_from TIMESTAMP,
    valid_to TIMESTAMP,
    is_current BOOLEAN,
    _processed_timestamp TIMESTAMP
)
USING DELTA
""")

""


In [23]:
spark.sql(f"""
MERGE INTO {SILVER_MENU_TABLE} AS target
USING menu_updates AS source
ON target.menu_item_id = source.menu_item_id AND target.is_current = true

WHEN MATCHED AND target.price != source.price THEN
  UPDATE SET
    target.valid_to = current_timestamp(),
    target.is_current = false

WHEN NOT MATCHED THEN
  INSERT (menu_item_id, item_name, price, valid_from, valid_to, is_current, _processed_timestamp)
  VALUES (source.menu_item_id, source.item_name, source.price, current_timestamp(), NULL, true, current_timestamp())
""")

,num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0,0


In [24]:
spark.sql(f"""
INSERT INTO {SILVER_MENU_TABLE} (menu_item_id, item_name, price, valid_from, valid_to, is_current, _processed_timestamp)
SELECT
    source.menu_item_id,
    source.item_name,
    source.price,
    current_timestamp(),
    NULL,
    true,
    current_timestamp()
FROM menu_updates AS source
WHERE NOT EXISTS (
    SELECT 1 FROM {SILVER_MENU_TABLE} AS target
    WHERE target.menu_item_id = source.menu_item_id
      AND target.is_current = true
      AND target.price = source.price
)
""")

,num_affected_rows,num_inserted_rows
0,0,0


In [25]:
spark.sql(f"""
MERGE INTO {SILVER_MENU_TABLE} AS target
USING menu_updates AS source
ON target.menu_item_id = source.menu_item_id AND target.is_current = true

WHEN MATCHED AND target.item_name != source.item_name THEN
  UPDATE SET target.item_name = source.item_name
""")

,num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0,0


In [26]:
spark.sql(f"SELECT * FROM {SILVER_MENU_TABLE} ORDER BY menu_item_id, valid_from").show(50)

+------------+--------------------+-----+--------------------+--------+----------+--------------------+
|menu_item_id|           item_name|price|          valid_from|valid_to|is_current|_processed_timestamp|
+------------+--------------------+-----+--------------------+--------+----------+--------------------+
|         101|           Hamburger|12.95|2026-08-05 00:54:...|    NULL|      true|2026-08-05 00:54:...|
|         102|        Cheeseburger|13.95|2026-08-05 00:54:...|    NULL|      true|2026-08-05 00:54:...|
|         103|             Hot Dog|  9.0|2026-08-05 00:54:...|    NULL|      true|2026-08-05 00:54:...|
|         104|       Veggie Burger| 10.5|2026-08-05 00:54:...|    NULL|      true|2026-08-05 00:54:...|
|         105|        Mac & Cheese|  7.0|2026-08-05 00:54:...|    NULL|      true|2026-08-05 00:54:...|
|         106|        French Fries|  7.0|2026-08-05 00:54:...|    NULL|      true|2026-08-05 00:54:...|
|         107|      Orange Chicken| 16.5|2026-08-05 00:54:...|  

In [27]:
df_bronze_menu.printSchema()

root
 |-- menu_item_id: integer (nullable = true)
 |-- item_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)

